# Comparatif des modeles sur les donnees du projet

Ce notebook genere un tableau comparatif similaire a ton exemple, calcule sur les donnees du projet.

Modeles compares :
- LR (grand volume)
- LR (petit volume)
- Tous les checkpoints Transformers detectes automatiquement (ex: DistilBERT, Mental-BERT, autres)

Metrices : recall (depressed), precision (depressed), accuracy, taille train, temps entrainement, interpretabilite, vitesse inference.

Le tableau se met a jour automatiquement: si un nouveau modele est ajoute dans models/fine_tuned, une nouvelle colonne est creee.

In [ ]:
import os
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

from src.training.preprocess import clean_text, load_kaggle_depression

pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 3)

print('Imports OK')

In [ ]:
# Configuration des chemins
DATA_CANDIDATES = [
    Path('data/raw'),
    Path('/content/drive/MyDrive/mh_detector/data'),
    Path('/content/mh_detector/data'),
]

def first_existing_path(paths):
    for p in paths:
        if p.exists():
            return p
    return None

DATA_DIR = first_existing_path(DATA_CANDIDATES)
if DATA_DIR is None:
    raise FileNotFoundError('Aucun dossier data trouve. Ajuste DATA_CANDIDATES.')

KAGGLE_PATH = DATA_DIR / 'reddit_depression_dataset.csv'
ERISK25_PATH = DATA_DIR / 'erisk25'

print('DATA_DIR      :', DATA_DIR)
print('KAGGLE_PATH   :', KAGGLE_PATH, '| exists =', KAGGLE_PATH.exists())
print('ERISK25_PATH  :', ERISK25_PATH, '| exists =', ERISK25_PATH.exists())

In [ ]:
# Chargeur eRisk25 robuste (evite les erreurs I/O type errno 107 sur Drive)
def load_erisk25_robust(data_path: Path, retries: int = 3) -> pd.DataFrame:
    json_dir = data_path / 'final-eriskt2-dataset-with-ground-truth' / 'all_combined'
    files = sorted(json_dir.glob('subject_*.json'))
    if not files:
        raise FileNotFoundError(f'Aucun fichier subject_*.json dans {json_dir}')

    rows = []
    skipped_os = 0
    skipped_json = 0

    for filepath in files:
        posts = None
        for attempt in range(1, retries + 1):
            try:
                with filepath.open('r', encoding='utf-8') as f:
                    posts = json.load(f)
                break
            except OSError:
                if attempt < retries:
                    time.sleep(0.6 * attempt)
                    continue
                skipped_os += 1
                posts = None
                break
            except json.JSONDecodeError:
                skipped_json += 1
                posts = None
                break

        if not posts:
            continue

        for post in posts:
            sub = post.get('submission', {})
            target = sub.get('target')
            if target is None:
                continue
            text = ((sub.get('title') or '') + ' ' + (sub.get('body') or '')).strip()
            if len(text) > 20:
                rows.append({'text': text, 'label': int(bool(target))})

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'eRisk25 vide. skipped_os={skipped_os}, skipped_json={skipped_json}')

    df['text'] = df['text'].apply(clean_text)
    df = df[df['text'].str.len() > 10].drop_duplicates(subset=['text']).reset_index(drop=True)

    print(f'eRisk25 : {len(df)} lignes | {df["label"].value_counts().to_dict()}')
    print(f'Fichiers ignores -> OSError: {skipped_os}, JSON: {skipped_json}')
    return df

In [ ]:
# Construction du dataset de travail
frames = []

if KAGGLE_PATH.exists():
    kaggle_df = load_kaggle_depression(KAGGLE_PATH, max_samples=100_000)
    print(f'Kaggle : {len(kaggle_df)} lignes | {kaggle_df["label"].value_counts().to_dict()}')
    frames.append(kaggle_df)

if ERISK25_PATH.exists():
    erisk_df = load_erisk25_robust(ERISK25_PATH)
    frames.append(erisk_df)

if not frames:
    raise RuntimeError('Aucune source chargee. Verifie KAGGLE_PATH et ERISK25_PATH.')

full_df = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['text']).reset_index(drop=True)
print('Dataset final :', len(full_df))
print('Distribution  :', full_df['label'].value_counts().to_dict())

# Split unique pour comparer tous les modeles sur le meme test set
train_df, test_df = train_test_split(
    full_df, test_size=0.2, random_state=42, stratify=full_df['label']
)

print('Train:', len(train_df), '| Test:', len(test_df))

In [ ]:
def format_train_size(n: int) -> str:
    if n >= 1_000_000:
        return f'{n/1_000_000:.1f}M'
    if n >= 1_000:
        return f'{n/1_000:.0f}K'
    return str(n)

def speed_bucket(samples_per_sec: float) -> str:
    if samples_per_sec >= 2000:
        return 'Very fast'
    if samples_per_sec >= 500:
        return 'Fast'
    if samples_per_sec >= 100:
        return 'Medium'
    return 'Slow'

def discover_transformer_models(search_roots: list[Path]) -> list[tuple[str, Path]]:
    discovered = []
    seen = set()

    for root in search_roots:
        if not root.exists():
            continue

        for cfg in root.rglob('config.json'):
            model_dir = cfg.parent
            has_tokenizer = (model_dir / 'tokenizer_config.json').exists()
            has_weights = (model_dir / 'model.safetensors').exists() or (model_dir / 'pytorch_model.bin').exists()
            if not (has_tokenizer and has_weights):
                continue

            key = str(model_dir.resolve())
            if key in seen:
                continue
            seen.add(key)

            display_name = model_dir.name.replace('_', '-').replace('checkpoint-', 'Checkpoint ')
            discovered.append((display_name, model_dir))

    discovered.sort(key=lambda x: x[0].lower())
    return discovered

def eval_lr(train_part: pd.DataFrame, test_part: pd.DataFrame, name: str) -> dict:
    start_train = time.perf_counter()
    model = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced'))
    ])
    model.fit(train_part['text'], train_part['label'])
    train_time = time.perf_counter() - start_train

    y_true = test_part['label']
    y_pred = model.predict(test_part['text'])

    # Temps d'inference sur un mini-batch representatif
    sample_texts = test_part['text'].iloc[: min(500, len(test_part))]
    start_inf = time.perf_counter()
    _ = model.predict(sample_texts)
    inf_time = max(time.perf_counter() - start_inf, 1e-9)
    sps = len(sample_texts) / inf_time

    return {
        'Model': name,
        'Recall (depressed)': recall_score(y_true, y_pred, pos_label=1),
        'Precision (depressed)': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Training data': format_train_size(len(train_part)),
        'Training time': f'{train_time:.1f}s',
        'Interpretable': 'Yes',
        'Inference speed': speed_bucket(sps),
    }

def eval_transformer_from_path(
    model_path: Path,
    train_size: int,
    test_part: pd.DataFrame,
    name: str
) -> dict:
    if not model_path.exists():
        return {
            'Model': name,
            'Recall (depressed)': np.nan,
            'Precision (depressed)': np.nan,
            'Accuracy': np.nan,
            'Training data': format_train_size(train_size),
            'Training time': 'N/A (modele absent)',
            'Interpretable': 'No',
            'Inference speed': 'N/A',
        }

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    model.eval()

    texts = test_part['text'].tolist()
    labels = test_part['label'].to_numpy()

    preds = []
    batch_size = 32

    start_inf = time.perf_counter()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())
    inf_time = max(time.perf_counter() - start_inf, 1e-9)

    sps = len(texts) / inf_time

    return {
        'Model': name,
        'Recall (depressed)': recall_score(labels, preds, pos_label=1),
        'Precision (depressed)': precision_score(labels, preds, pos_label=1, zero_division=0),
        'Accuracy': accuracy_score(labels, preds),
        'Training data': format_train_size(train_size),
        'Training time': 'Voir log entrainement',
        'Interpretable': 'No',
        'Inference speed': speed_bucket(sps),
    }

In [ ]:
# Experiences
SMALL_TRAIN_SIZE = 21_000

results = []

# 1) LR sur grand volume
results.append(eval_lr(train_df, test_df, name=f'LR ({format_train_size(len(train_df))})'))

# 2) LR sur petit volume
train_small, _ = train_test_split(
    train_df,
    train_size=min(SMALL_TRAIN_SIZE, len(train_df)),
    random_state=42,
    stratify=train_df['label']
)
results.append(eval_lr(train_small, test_df, name=f'LR ({format_train_size(len(train_small))})'))

# 3) Detection automatique des checkpoints Transformers
MODEL_SEARCH_ROOTS = [
    Path('models/fine_tuned'),
    Path('models'),
]
detected_models = discover_transformer_models(MODEL_SEARCH_ROOTS)

print(f'Modeles Transformers detectes: {len(detected_models)}')
for model_name, model_path in detected_models:
    display_name = f'{model_name} ({format_train_size(len(train_small))})'
    print(' -', display_name, '->', model_path)
    results.append(
        eval_transformer_from_path(
            model_path=model_path,
            train_size=len(train_small),
            test_part=test_df,
            name=display_name,
        )
    )

if len(detected_models) == 0:
    print('Aucun checkpoint Transformers detecte. Ajoute un modele dans models/fine_tuned pour l inclure automatiquement.')

results

In [ ]:
# Tableau final (style proche de ton exemple)
table_df = pd.DataFrame(results).set_index('Model').T

# Arrondis visuels
for row_name in ['Recall (depressed)', 'Precision (depressed)', 'Accuracy']:
    table_df.loc[row_name] = pd.to_numeric(table_df.loc[row_name], errors='coerce').round(3)

display(table_df)

# Export optionnel
Path('reports').mkdir(exist_ok=True)
table_df.to_csv('reports/model_comparison_table.csv')
print('Tableau exporte -> reports/model_comparison_table.csv')

## Notes

- Le tableau ajoute automatiquement les checkpoints Transformers trouves dans models/fine_tuned et models.
- Si aucun checkpoint valide n est detecte, seules les colonnes LR sont affichees.
- Pour comparer un nouveau modele, il suffit de sauvegarder son dossier (config.json + tokenizer_config.json + poids) dans models/fine_tuned puis relancer les cellules 6, 7 et 8.